# Model diagnostics and interpretation

What to look at after the metric, each check on a handful of rows first.

**What's in here**
- error table, bias, error by group, worst rows
- rolling RMSE over time
- train vs test gap (overfitting)
- coefficient stability across periods
- correlation matrix, VIF by hand, what collinearity does to OLS vs Ridge
- permutation importance by hand and with sklearn
- partial dependence by hand
- residual autocorrelation
- the leaked-feature detector
- the same checks on a real lag model
- how to critique a model in five minutes

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

## 1. Error table and bias

Six rows of true value, prediction, error. `error = y - pred`, positive means under-prediction.

In [2]:
res = pd.DataFrame({"hour": [0, 1, 0, 1, 0, 1],
                    "y":    [10, 20, 11, 22, 12, 24],
                    "pred": [10, 18, 11, 19, 12, 21]})
res["error"] = res["y"] - res["pred"]
res

,hour,y,pred,error
0,0,10,10,0
1,1,20,18,2
2,0,11,11,0
3,1,22,19,3
4,0,12,12,0
5,1,24,21,3


In [3]:
print("mean error (bias):", round(res["error"].mean(), 2))
print("RMSE             :", round(np.sqrt((res["error"] ** 2).mean()), 2))
res["error"].describe().round(2)

mean error (bias): 1.33
RMSE             : 1.91


count    6.00
mean     1.33
std      1.51
min      0.00
25%      0.00
50%      1.00
75%      2.75
max      3.00
Name: error, dtype: float64

In [4]:
res.groupby("hour")["error"].agg(["mean", "count"])

,mean,count
hour,,
0,0.000000,3
1,2.666667,3


The bias hides in hour 1. `nlargest` on the absolute error finds the worst rows.

In [5]:
res.loc[res["error"].abs().nlargest(2).index]

,hour,y,pred,error
3,1,22,19,3
5,1,24,21,3


## 2. Rolling RMSE over time

Square the errors, take a rolling mean, take the square root. Window of 3 on 6 rows.

In [6]:
e = pd.Series([1, -1, 2, 5, -6, 4], name="error")
pd.DataFrame({"error": e, "error2": e ** 2, "rolling3_mse": (e ** 2).rolling(3).mean(), "rolling3_rmse": np.sqrt((e ** 2).rolling(3).mean()).round(2)})

,error,error2,rolling3_mse,rolling3_rmse
0,1,1,NaN,NaN
1,-1,1,NaN,NaN
2,2,4,2.000000,1.41
3,5,25,10.000000,3.16
4,-6,36,21.666667,4.65
5,4,16,25.666667,5.07


Rows 3-5 have larger errors and the rolling RMSE rises from 1.4 to 5.1: the model got worse over time.

## 3. Train vs test gap

Five training points on a line plus noise, one test point. A degree-4 polynomial fits the
training points exactly and misses the test point.

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

Xt = pd.DataFrame({"x": [1, 2, 3, 4, 5]})
yt = pd.Series([2.1, 3.9, 6.2, 7.8, 10.1])       # roughly 2x
X_new = pd.DataFrame({"x": [6]})
y_new = 12.0
for deg in [1, 4]:
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(Xt, yt)
    train_rmse = np.sqrt(((m.predict(Xt) - yt) ** 2).mean())
    print(f"degree {deg}: train RMSE {train_rmse:.3f}   prediction at x=6: {m.predict(X_new)[0]:.2f}  (true 12)")

degree 1: train RMSE 0.146   prediction at x=6: 11.99  (true 12)
degree 4: train RMSE 0.000   prediction at x=6: 17.10  (true 12)


Degree 4: train RMSE 0, prediction far off. A train score much better than the test score is
overfitting. Train and test both near-perfect is usually leakage.

## 4. Coefficient stability

Fit the same model on the first half and the second half. If the coefficients move a lot,
the relationship is not stable (or the features are collinear).

In [8]:
Xs = pd.DataFrame({"x1": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]})
ys = pd.Series([2, 4, 6, 8, 10, 15, 18, 21, 24, 27])     # slope 2 first half, slope 3 second half
first = LinearRegression().fit(Xs.iloc[:5], ys.iloc[:5])
second = LinearRegression().fit(Xs.iloc[5:], ys.iloc[5:])
pd.DataFrame({"first_half": [first.coef_[0], first.intercept_], "second_half": [second.coef_[0], second.intercept_]}, index=["coef x1", "intercept"]).round(2)

,first_half,second_half
coef x1,2.0,3.0
intercept,0.0,-3.0


## 5. Collinearity and VIF

Three columns; `x2` is `x1` plus a little noise. The correlation matrix shows it.

In [9]:
Xc = pd.DataFrame({"x1": [1, 2, 3, 4, 5, 6],
                   "x2": [1.1, 1.9, 3.2, 3.9, 5.1, 6.0],
                   "x3": [3, 1, 4, 1, 5, 9]})
Xc.corr().round(3)

,x1,x2,x3
x1,1.000,0.998,0.696
x2,0.998,1.000,0.721
x3,0.696,0.721,1.000


VIF for a column = 1 / (1 − R²) where R² comes from regressing that column on the other columns.

In [10]:
for col in Xc.columns:
    others = Xc.drop(columns=col)
    r2 = LinearRegression().fit(others, Xc[col]).score(others, Xc[col])
    print(f"{col}: R2 on others = {r2:.3f}   VIF = {1 / (1 - r2):.1f}")

x1: R2 on others = 0.997   VIF = 331.6
x2: R2 on others = 0.997   VIF = 356.3
x3: R2 on others = 0.655   VIF = 2.9


VIF above 10 is a warning. `y` depends on x1 only, but OLS spreads the effect between x1 and x2
in a way that is unstable; Ridge shares it evenly.

In [11]:
from sklearn.linear_model import Ridge

yc = 2 * Xc["x1"] + pd.Series([0.1, -0.1, 0.2, -0.2, 0.1, -0.1])
ols = LinearRegression().fit(Xc, yc)
rdg = Ridge(alpha=1.0).fit(Xc, yc)
pd.DataFrame({"OLS": ols.coef_, "Ridge(1)": rdg.coef_}, index=Xc.columns).round(2)

,OLS,Ridge(1)
x1,0.75,0.95
x2,1.24,0.92
x3,-0.00,0.05


## 6. Permutation importance by hand

Fit, score, then shuffle one column and score again. The drop is that column's importance.

In [12]:
Xp = pd.DataFrame({"x1": [1, 2, 3, 4, 5, 6], "x2": [5, 3, 8, 1, 9, 2]})
yp = 2 * Xp["x1"] + 1
m = LinearRegression().fit(Xp, yp)
print("R2 original          :", round(m.score(Xp, yp), 3))

Xp_shuf = Xp.copy()
Xp_shuf["x1"] = [6, 1, 4, 2, 5, 3]              # x1 shuffled by hand
print("R2 with x1 shuffled  :", round(m.score(Xp_shuf, yp), 3))

Xp_shuf2 = Xp.copy()
Xp_shuf2["x2"] = [2, 9, 1, 8, 3, 5]             # x2 shuffled
print("R2 with x2 shuffled  :", round(m.score(Xp_shuf2, yp), 3))

R2 original          : 1.0
R2 with x1 shuffled  : -1.286
R2 with x2 shuffled  : 1.0


In [13]:
from sklearn.inspection import permutation_importance

pi = permutation_importance(m, Xp, yp, n_repeats=5, random_state=0)
pd.Series(pi.importances_mean, index=Xp.columns).round(3)

x1    1.76
x2    0.00
dtype: float64

Shuffling x1 destroys the score; shuffling x2 changes nothing (it has coefficient 0).

## 7. Partial dependence by hand

Vary one feature over a grid while holding the others at their median, predict, and read the curve.

In [14]:
grid = pd.DataFrame({"x1": [1, 3, 5], "x2": Xp["x2"].median()})
grid["pred"] = m.predict(grid)
grid

,x1,x2,pred
0,1,4.0,3.0
1,3,4.0,7.0
2,5,4.0,11.0


## 8. Residual autocorrelation

If today's error predicts tomorrow's, the model is missing a lag. `autocorr(1)` is the
correlation of the error with its own previous value.

In [15]:
err = pd.Series([1.0, 1.2, 0.9, -1.0, -1.1, -0.8, 1.1, 0.9])
pd.DataFrame({"error": err, "error_lag1": err.shift(1)})

,error,error_lag1
0,1.0,NaN
1,1.2,1.0
2,0.9,1.2
3,-1.0,0.9
4,-1.1,-1.0
5,-0.8,-1.1
6,1.1,-0.8
7,0.9,1.1


In [16]:
print("autocorr(1):", round(err.autocorr(1), 3))

autocorr(1): 0.464


0.46 means the errors come in runs (three positives, three negatives, then positives). For a 1-hour-ahead model that usually means "add lag 1"
or "the model barely beats persistence".

## 9. The leaked-feature detector

Add a column that is a copy of `y`. The model finds it immediately: coefficient 1, R² 1.

In [17]:
Xl = pd.DataFrame({"x1": [1, 2, 3, 4, 5], "leak": [5, 8, 8, 11, 14]})
yl = pd.Series([5, 8, 8, 11, 14])
ml = LinearRegression().fit(Xl, yl)
print("coefs:", dict(zip(Xl.columns, ml.coef_.round(3))))
print("R2   :", ml.score(Xl, yl))

coefs: {'x1': 0.0, 'leak': 1.0}
R2   : 1.0


The subtle version: a rolling mean that includes the current value. On 5 rows, `rolling(2).mean()`
at row 3 is (y[2] + y[3]) / 2, so `y[3] = 2 * roll - y[2]`: the target can be recovered exactly.

In [18]:
yy = pd.Series([10, 12, 11, 15, 14], name="y")
feat = pd.DataFrame({"roll2": yy.rolling(2).mean(), "lag1": yy.shift(1), "roll2_shifted": yy.shift(1).rolling(2).mean()})
pd.concat([yy, feat], axis=1)

,y,roll2,lag1,roll2_shifted
0,10,NaN,NaN,NaN
1,12,11.0,10.0,NaN
2,11,11.5,12.0,11.0
3,15,13.0,11.0,11.5
4,14,14.5,15.0,13.0


In [19]:
ok = feat.dropna()
ml = LinearRegression().fit(ok[["roll2", "lag1"]], yy[ok.index])
print("coefs on [roll2, lag1]:", ml.coef_.round(2), " R2:", round(ml.score(ok[["roll2", "lag1"]], yy[ok.index]), 3))

coefs on [roll2, lag1]: [ 2. -1.]  R2: 1.0


Coefficients (2, −1) and R² = 1: the model reconstructs `y` from the leak. With
`shift(1).rolling(2)` this cannot happen.

## 10. On a real lag model

Ridge on consumption with lags 1, 24, 168, a shifted rolling mean, temperature and hour dummies.

In [20]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).sort_values("time").reset_index(drop=True)
y = df["consumption_mwh"]
X = pd.DataFrame({"lag1": y.shift(1), "lag24": y.shift(24), "lag168": y.shift(168),
                  "roll24": y.shift(1).rolling(24).mean(), "temp_c": df["temp_c"]})
X = pd.concat([X, pd.get_dummies(df["time"].dt.hour, prefix="h", drop_first=True, dtype=int)], axis=1)
keep = X.dropna().index
X, y, t = X.loc[keep], y.loc[keep], df.loc[keep, "time"]
split = int(len(X) * 0.8)
model = Ridge(alpha=1.0).fit(X.iloc[:split], y.iloc[:split])
out = pd.DataFrame({"time": t.iloc[split:], "y": y.iloc[split:], "pred": model.predict(X.iloc[split:])})
out["error"] = out["y"] - out["pred"]
print("train R2:", round(model.score(X.iloc[:split], y.iloc[:split]), 4), " test R2:", round(model.score(X.iloc[split:], y.iloc[split:]), 4))
out.head(3)

train R2: 0.9834  test R2: 0.9802


,time,y,pred,error
14049,2023-08-09 09:00:00+00:00,29182.0,28762.873509,419.126491
14050,2023-08-09 10:00:00+00:00,29478.4,29105.966282,372.433718
14051,2023-08-09 11:00:00+00:00,29436.4,29143.545638,292.854362


In [21]:
print("bias:", round(out["error"].mean(), 1))
out.groupby(out["time"].dt.hour)["error"].mean().round(0).to_frame("mean_error").T

bias: -4.8


time,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
mean_error,20.0,47.0,42.0,-49.0,-13.0,66.0,-33.0,45.0,-32.0,-4.0,-66.0,15.0,-33.0,-6.0,-67.0,-72.0,32.0,29.0,8.0,64.0,-46.0,32.0,-111.0,19.0


In [22]:
out.groupby(out["time"].dt.month)["error"].apply(lambda e: np.sqrt((e ** 2).mean())).round(0).rename("rmse_by_month")

time
8     524.0
9     555.0
10    551.0
11    555.0
12    565.0
Name: rmse_by_month, dtype: float64

In [23]:
out.loc[out["error"].abs().nlargest(5).index]

,time,y,pred,error
15168,2023-09-25 00:00:00+00:00,25756.6,22828.822272,2927.777728
15336,2023-10-02 00:00:00+00:00,24973.8,22151.066771,2822.733229
15000,2023-09-18 00:00:00+00:00,25220.9,22668.435202,2552.464798
16128,2023-11-04 00:00:00+00:00,24388.0,26845.822325,-2457.822325
15120,2023-09-23 00:00:00+00:00,20460.5,22769.973513,-2309.473513


The worst hours are at 00:00: the model uses lag 1 heavily and the day boundary between weekend
and weekday levels catches it out. That is the "see result, investigate why" step.

In [24]:
pd.Series(model.coef_[:5], index=X.columns[:5]).round(3)

lag1       0.880
lag24      0.007
lag168     0.061
roll24    -0.021
temp_c   -26.067
dtype: float64

In [25]:
lagcols = ["lag1", "lag24", "lag168", "roll24", "temp_c"]
for col in lagcols:
    others = X[lagcols].drop(columns=col)
    r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
    print(f"{col:7s} VIF = {1 / (1 - r2):6.1f}")

lag1    VIF =    5.0


lag24   VIF =    6.2
lag168  VIF =    6.4
roll24  VIF =    4.4
temp_c  VIF =    3.4


In [26]:
pi = permutation_importance(model, X.iloc[split:], y.iloc[split:], n_repeats=3, random_state=0)
pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False).head(5).round(3)

lag1    1.547
h_17    0.068
h_7     0.066
h_16    0.045
h_6     0.042
dtype: float64

In [27]:
print("residual autocorr(1):", round(out["error"].autocorr(1), 3))
naive = y.shift(1).iloc[split:]
print("naive lag-1 R2      :", round(1 - ((y.iloc[split:] - naive) ** 2).sum() / ((y.iloc[split:] - y.iloc[split:].mean()) ** 2).sum(), 4))

residual autocorr(1): 0.0
naive lag-1 R2      : 0.8495


Persistence alone gives R² 0.85; the model gives 0.98, so the lags and hour dummies add real
skill. Residual autocorrelation near 0 says there is no obvious missing lag. Permutation
importance confirms lag 1 carries most of it.

## How to critique a model in five minutes

1. Print the metric next to the naive baseline on the same rows.
2. Train vs test score.
3. Mean error by hour / weekday / month.
4. The 5 worst rows and what they have in common.
5. Rolling RMSE: is the error stable over time?
6. Coefficients with names; anything near 1 on a target-derived feature?
7. Residual autocorrelation.

## Quick reference

| Check | Code |
|---|---|
| bias | `(y - pred).mean()` |
| error by group | `out.groupby(key)["error"].mean()` |
| worst rows | `out.loc[out.error.abs().nlargest(5).index]` |
| rolling RMSE | `np.sqrt((err**2).rolling(w).mean())` |
| VIF | `1 / (1 - R2_of_col_on_others)` |
| importance | `permutation_importance(model, X_test, y_test)` |
| autocorrelation | `err.autocorr(1)` |
| leak | coefficient ≈ 1 or (2, −1) pattern, R² ≈ 1 |